<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_06-5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 실습 5: RAG Context Builder 만들기

이번 실습에서는 검색된 문서 조각(chunk)을 그대로 LLM에 넣지 않고,
**질문과 관련된 근거만 골라 Context를 만드는 과정**을 구현합니다.

또한 간단한 **Contextual Retrieval 아이디어**를 적용하여,
각 chunk에 짧은 문맥 설명을 붙인 뒤 검색 결과가 어떻게 달라지는지 비교합니다.

### 학습 목표
- Retrieval 결과와 LLM Context의 차이를 이해한다.
- 검색된 chunk 중 필요한 것만 선택하는 Context Builder를 만든다.
- Contextual Retrieval의 기본 아이디어를 이해한다.
- 불필요한 문서를 줄이면 Context 품질이 좋아질 수 있음을 확인한다.

> 이 실습은 개념 이해를 위한 간단한 예제입니다.  
> 실제 서비스에서는 reranker, metadata filter, access control, token budget 등을 함께 고려합니다.

## 1. 실습용 문서 준비

이번에는 하나의 대학 강의 문서를 여러 chunk로 나누었다고 가정합니다.

In [1]:
chunks = {
    "c1": "지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.",
    "c2": "평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.",
    "c3": "출석은 전체 수업의 4분의 1 이상 결석하면 성적에 불이익이 있을 수 있다.",
    "c4": "과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.",
    "c5": "RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 지표를 순서대로 학습한다.",
    "c6": "강의 자료는 수업 전에 LMS에 업로드하며, 실습 파일은 별도로 제공한다."
}

for cid, text in chunks.items():
    print(cid, ":", text)

c1 : 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.
c2 : 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
c3 : 출석은 전체 수업의 4분의 1 이상 결석하면 성적에 불이익이 있을 수 있다.
c4 : 과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.
c5 : RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 지표를 순서대로 학습한다.
c6 : 강의 자료는 수업 전에 LMS에 업로드하며, 실습 파일은 별도로 제공한다.


## 2. 질문 준비

사용자 질문은 다음과 같습니다.

**“지능시스템 과목의 과제 비율은 몇 퍼센트인가?”**

이 질문에 답하려면 평가 비율이 들어 있는 chunk가 가장 중요합니다.

In [2]:
query = "지능시스템 과목의 과제 비율은 몇 퍼센트인가?"

print("질문:")
print(query)

질문:
지능시스템 과목의 과제 비율은 몇 퍼센트인가?


## 3. TF-IDF 기반 간단한 검색기 만들기

검색의 핵심만 보기 위해 TF-IDF + Cosine Similarity를 사용합니다.

In [3]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def tokenize_korean(text):
    """
    쉬운 실습용 tokenizer
    - 영문은 소문자로 변환
    - 한글/영문/숫자만 남김
    - 공백 기준으로 분리
    """
    text = text.lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣%]+", " ", text)
    return text.split()


chunk_ids = list(chunks.keys())
chunk_texts = list(chunks.values())

vectorizer = TfidfVectorizer(
    tokenizer=tokenize_korean,
    token_pattern=None,
    lowercase=False
)

X = vectorizer.fit_transform(chunk_texts)

print("문서 수:", X.shape[0])
print("TF-IDF 차원 수:", X.shape[1])

문서 수: 6
TF-IDF 차원 수: 59


## 4. 검색 함수 만들기

질문과 각 chunk의 cosine similarity를 계산하여
점수가 높은 순서대로 정렬합니다.

In [4]:
def retrieve(query, top_k=4):
    """
    질문과 관련된 상위 top_k chunk를 검색합니다.
    """

    # 질문을 기존 TF-IDF 공간으로 변환
    q = vectorizer.transform([query])

    # cosine similarity 계산
    scores = cosine_similarity(q, X)[0]

    # 높은 점수 순으로 정렬
    ranking = np.argsort(-scores)

    results = []

    for idx in ranking[:top_k]:
        results.append({
            "chunk_id": chunk_ids[idx],
            "score": float(scores[idx]),
            "text": chunks[chunk_ids[idx]]
        })

    return results

In [5]:
retrieved = retrieve(query, top_k=4)

print("질문:", query)
print()

for rank, item in enumerate(retrieved, 1):
    print(
        rank,
        item["chunk_id"],
        f"score={item['score']:.4f}"
    )
    print(" ", item["text"])
    print()

질문: 지능시스템 과목의 과제 비율은 몇 퍼센트인가?

1 c2 score=0.3380
  평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.

2 c1 score=0.1961
  지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.

3 c3 score=0.0000
  출석은 전체 수업의 4분의 1 이상 결석하면 성적에 불이익이 있을 수 있다.

4 c4 score=0.0000
  과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.



## 5. 문제: 검색 결과를 전부 Context에 넣으면?

검색 결과 상위 4개를 그대로 LLM Context로 넣는다고 가정해 봅니다.

이 방법은 간단하지만, 질문과 직접 관련 없는 내용까지 함께 들어갈 수 있습니다.

In [6]:
def build_naive_context(retrieved_results):
    """
    검색된 chunk를 모두 이어 붙이는 가장 단순한 Context Builder
    """

    context_parts = []

    for item in retrieved_results:
        context_parts.append(
            f"[{item['chunk_id']}] {item['text']}"
        )

    return "\n".join(context_parts)


naive_context = build_naive_context(retrieved)

print("=== Naive Context ===")
print(naive_context)

=== Naive Context ===
[c2] 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
[c1] 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.
[c3] 출석은 전체 수업의 4분의 1 이상 결석하면 성적에 불이익이 있을 수 있다.
[c4] 과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.


### 관찰 포인트

질문은 **과제 비율**만 묻고 있지만,
검색 결과를 모두 넣으면 수업 소개, 출석, 실습 안내 등이 함께 들어갈 수 있습니다.

즉,

**Retrieved Documents ≠ Final Context**

검색 결과 중에서도 질문에 필요한 근거를 골라야 합니다.

## 6. 간단한 Context Builder 만들기

이번에는 다음 기준을 사용합니다.

- 검색 점수가 일정 값 이상인 chunk만 사용
- 너무 많은 chunk는 넣지 않음
- 질문과 직접 관련된 내용만 Context에 포함

실무에서는 이 단계에 reranker나 LLM 기반 relevance 판단을 사용할 수 있습니다.

In [7]:
def build_context(
    retrieved_results,
    min_score=0.05,
    max_chunks=2
):
    """
    검색 결과 중 점수가 높은 chunk만 골라 Context를 만듭니다.

    min_score:
        너무 관련성이 낮은 문서를 제외하는 기준

    max_chunks:
        Context에 넣을 최대 chunk 수
    """

    selected = []

    for item in retrieved_results:

        # 관련성이 너무 낮으면 제외
        if item["score"] < min_score:
            continue

        selected.append(item)

        # 최대 개수에 도달하면 중단
        if len(selected) >= max_chunks:
            break

    context = "\n".join(
        f"[{item['chunk_id']}] {item['text']}"
        for item in selected
    )

    return selected, context

In [8]:
selected, context = build_context(
    retrieved,
    min_score=0.05,
    max_chunks=2
)

print("=== 선택된 Chunk ===")

for item in selected:
    print(
        item["chunk_id"],
        f"score={item['score']:.4f}",
        item["text"]
    )

print()
print("=== Final Context ===")
print(context)

=== 선택된 Chunk ===
c2 score=0.3380 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
c1 score=0.1961 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.

=== Final Context ===
[c2] 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
[c1] 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.


## 7. 질문-근거-답변 구조 확인

현재 질문에 필요한 핵심 근거는 다음 문장입니다.

> 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.

따라서 LLM에는 가능하면 이 근거가 중심이 된 Context를 전달하는 것이 좋습니다.

In [9]:
answer = "과제 비율은 20%입니다."

print("질문:", query)
print()
print("Context:")
print(context)
print()
print("예상 답변:")
print(answer)

질문: 지능시스템 과목의 과제 비율은 몇 퍼센트인가?

Context:
[c2] 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
[c1] 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.

예상 답변:
과제 비율은 20%입니다.


# 8. Contextual Retrieval 아이디어

일반적인 chunk는 원문 일부만 가지고 있기 때문에
문서 전체 맥락이 사라질 수 있습니다.

Contextual Retrieval에서는 각 chunk에
**이 chunk가 전체 문서에서 어떤 의미를 가지는지 짧은 설명**을 붙여 검색에 사용합니다.

예:

원래 chunk:

> 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.

문맥을 붙인 chunk:

> [지능시스템 과목의 성적 평가 기준]  
> 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.

이렇게 하면 검색어와의 연결이 더 쉬워질 수 있습니다.

## 9. 각 chunk에 짧은 문맥 설명 붙이기

실제 Contextual Retrieval에서는 LLM을 이용해 문맥 설명을 자동 생성할 수 있습니다.

이번 실습에서는 원리를 보기 위해 사람이 만든 짧은 설명을 사용합니다.

In [10]:
context_labels = {
    "c1": "지능시스템 과목의 수업 소개",
    "c2": "지능시스템 과목의 성적 평가 기준과 과제 비율",
    "c3": "지능시스템 과목의 출석 규정",
    "c4": "지능시스템 과목의 과제와 실습 환경",
    "c5": "지능시스템 과목의 RAG 실습 내용",
    "c6": "지능시스템 과목의 강의 자료 제공 방식"
}

contextual_chunks = {}

for cid, text in chunks.items():
    contextual_chunks[cid] = (
        f"[{context_labels[cid]}] {text}"
    )

for cid, text in contextual_chunks.items():
    print(cid, ":", text)

c1 : [지능시스템 과목의 수업 소개] 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.
c2 : [지능시스템 과목의 성적 평가 기준과 과제 비율] 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
c3 : [지능시스템 과목의 출석 규정] 출석은 전체 수업의 4분의 1 이상 결석하면 성적에 불이익이 있을 수 있다.
c4 : [지능시스템 과목의 과제와 실습 환경] 과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.
c5 : [지능시스템 과목의 RAG 실습 내용] RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 지표를 순서대로 학습한다.
c6 : [지능시스템 과목의 강의 자료 제공 방식] 강의 자료는 수업 전에 LMS에 업로드하며, 실습 파일은 별도로 제공한다.


## 10. Contextual Chunk로 검색기 다시 만들기

In [11]:
contextual_texts = [
    contextual_chunks[cid]
    for cid in chunk_ids
]

contextual_vectorizer = TfidfVectorizer(
    tokenizer=tokenize_korean,
    token_pattern=None,
    lowercase=False
)

X_contextual = contextual_vectorizer.fit_transform(
    contextual_texts
)


def retrieve_contextual(query, top_k=4):
    """
    문맥 설명이 붙은 chunk를 대상으로 검색합니다.
    """

    q = contextual_vectorizer.transform([query])

    scores = cosine_similarity(
        q,
        X_contextual
    )[0]

    ranking = np.argsort(-scores)

    results = []

    for idx in ranking[:top_k]:
        cid = chunk_ids[idx]

        results.append({
            "chunk_id": cid,
            "score": float(scores[idx]),
            "text": chunks[cid],
            "contextual_text": contextual_chunks[cid]
        })

    return results

In [12]:
contextual_results = retrieve_contextual(
    query,
    top_k=4
)

print("질문:", query)
print()

for rank, item in enumerate(contextual_results, 1):
    print(
        rank,
        item["chunk_id"],
        f"score={item['score']:.4f}"
    )
    print(" ", item["contextual_text"])
    print()

질문: 지능시스템 과목의 과제 비율은 몇 퍼센트인가?

1 c2 score=0.4935
  [지능시스템 과목의 성적 평가 기준과 과제 비율] 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.

2 c1 score=0.1189
  [지능시스템 과목의 수업 소개] 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.

3 c5 score=0.0700
  [지능시스템 과목의 RAG 실습 내용] RAG 실습에서는 Chunking, BM25, Dense Retrieval, 평가 지표를 순서대로 학습한다.

4 c4 score=0.0684
  [지능시스템 과목의 과제와 실습 환경] 과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.



## 11. 일반 Retrieval과 Contextual Retrieval 비교

같은 질문에 대해 두 검색 결과를 나란히 비교합니다.

In [13]:
normal_results = retrieve(query, top_k=4)
context_results = retrieve_contextual(query, top_k=4)

rows = []

for rank in range(4):
    rows.append({
        "Rank": rank + 1,
        "일반 Retrieval": normal_results[rank]["chunk_id"],
        "일반 점수": round(normal_results[rank]["score"], 4),
        "Contextual Retrieval": context_results[rank]["chunk_id"],
        "Contextual 점수": round(context_results[rank]["score"], 4)
    })

comparison = pd.DataFrame(rows)

comparison

,Rank,일반 Retrieval,일반 점수,Contextual Retrieval,Contextual 점수
0,1,c2,0.3380,c2,0.4935
1,2,c1,0.1961,c1,0.1189
2,3,c3,0.0000,c5,0.0700
3,4,c4,0.0000,c4,0.0684


### 중요한 해석

Contextual Retrieval의 목적은

- 원래 chunk만 보면 부족한 문맥을 보완하고
- 검색 질의와의 연결을 더 명확하게 만들어
- 관련 문서가 검색될 가능성을 높이는 것

입니다.

작은 예제에서는 순위가 같을 수도 있습니다.
실제 효과는 데이터, 질의, chunk 품질, tokenizer, 검색 방식에 따라 달라집니다.

# 12. 간단한 RAG Prompt 만들기

이제 선택된 Context를 LLM에 전달할 Prompt 형태로 만들어 봅니다.

중요한 규칙:

- Context에 있는 정보만 사용
- 근거가 없으면 모른다고 답하기
- 짧고 직접적으로 답하기

In [14]:
def make_rag_prompt(query, context):
    """
    RAG에서 사용할 간단한 Prompt를 만듭니다.
    """

    prompt = f"""
다음 Context만 사용하여 질문에 답하세요.

규칙:
1. Context에 있는 사실만 사용하세요.
2. 근거가 없으면 "제공된 문서에서 확인할 수 없습니다."라고 답하세요.
3. 핵심만 간단히 답하세요.

[Context]
{context}

[Question]
{query}

[Answer]
"""

    return prompt.strip()

In [15]:
prompt = make_rag_prompt(
    query,
    context
)

print(prompt)

다음 Context만 사용하여 질문에 답하세요.

규칙:
1. Context에 있는 사실만 사용하세요.
2. 근거가 없으면 "제공된 문서에서 확인할 수 없습니다."라고 답하세요.
3. 핵심만 간단히 답하세요.

[Context]
[c2] 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
[c1] 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.

[Question]
지능시스템 과목의 과제 비율은 몇 퍼센트인가?

[Answer]


# 13. 문서에 없는 질문 처리하기

RAG 시스템에서는 **문서에 없는 질문**도 중요합니다.

예를 들어:

> “지능시스템 과목의 팀 프로젝트 비율은 몇 퍼센트인가?”

현재 문서에는 팀 프로젝트 비율 정보가 없습니다.

이 경우 관련 없는 chunk를 억지로 사용하여 답하면 안 됩니다.

In [16]:
unknown_query = "지능시스템 과목의 팀 프로젝트 비율은 몇 퍼센트인가?"

unknown_results = retrieve(
    unknown_query,
    top_k=4
)

for rank, item in enumerate(unknown_results, 1):
    print(
        rank,
        item["chunk_id"],
        f"score={item['score']:.4f}",
        item["text"]
    )

1 c1 score=0.2401 지능시스템 과목은 자연어처리, RAG, 에이전트를 다루는 4학년 전공 수업이다.
2 c2 score=0.2070 평가 비율은 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%이다.
3 c3 score=0.0000 출석은 전체 수업의 4분의 1 이상 결석하면 성적에 불이익이 있을 수 있다.
4 c4 score=0.0000 과제는 Colab 환경에서 Python으로 수행하며 일부 실습은 Hugging Face 모델을 사용한다.


## 14. 최소 관련성 기준을 이용해 답변 보류하기

검색 점수가 너무 낮으면
“근거 부족”으로 판단하는 간단한 방법을 사용할 수 있습니다.

> 실제 시스템에서는 score threshold 하나만으로 충분하지 않을 수 있으며,
> reranker, classifier, LLM judge 등의 추가 검증을 사용할 수 있습니다.

In [22]:
def safe_context_builder(
    query,
    top_k=4,
    min_score=0.10,
    max_chunks=2
):
    """
    질문을 검색한 뒤,
    일정 점수 이상의 chunk가 있을 때만 Context를 만듭니다.
    """

    results = retrieve(
        query,
        top_k=top_k
    )

    selected = [
        item
        for item in results
        if item["score"] >= min_score
    ][:max_chunks]

    if len(selected) == 0:
        return [], ""

    context = "\n".join(
        f"[{item['chunk_id']}] {item['text']}"
        for item in selected
    )

    return selected, context

In [23]:
selected_unknown, unknown_context = safe_context_builder(
    unknown_query,
    min_score=0.30
)

print("질문:", unknown_query)
print()

if unknown_context:
    print("선택된 Context:")
    print(unknown_context)
else:
    print("충분한 근거를 찾지 못했습니다.")
    print("예상 답변: 제공된 문서에서 확인할 수 없습니다.")

질문: 지능시스템 과목의 팀 프로젝트 비율은 몇 퍼센트인가?

충분한 근거를 찾지 못했습니다.
예상 답변: 제공된 문서에서 확인할 수 없습니다.


# 핵심 정리

### Retrieval
관련 문서 후보를 찾는 단계

### Context Builder
검색 결과 중 실제 답변에 필요한 근거를 골라 LLM 입력으로 만드는 단계

### Contextual Retrieval
chunk만으로 부족한 문맥을 보완하여 검색 가능성을 높이는 방법

---

> **좋은 RAG는 많이 넣는 시스템이 아니라, 질문에 필요한 근거를 잘 골라 넣는 시스템입니다.**